# SQL Aggregation Performance Tests

This notebook benchmarks the Rust-based SQL execution engine (data-kernel) with 1 million orders and ~5.5 million order items.

## Test Environment
- **Dataset**: 1M orders, 5.5M order_items
- **Engine**: DataFusion + WGPU (GPU-accelerated)
- **Test Types**: Simple aggregations, GROUP BY, JOINs, filtered queries
- **API**: Python data_kernel package


## Setup and Dependencies


In [ ]:
import os
import time
from pathlib import Path
import pandas as pd
from data_kernel import execute, is_gpu_available, get_gpu_info

# Configuration
DATA_DIR = Path.cwd() / "data"

# Set DATA_PATH for the executor to find parquet files
os.environ['DATA_PATH'] = str(DATA_DIR.absolute())

# Results storage
results = []

print("✓ Imports loaded")
print(f"✓ Data directory: {DATA_DIR}")
print(f"✓ DATA_PATH set to: {os.environ['DATA_PATH']}")


## GPU Availability Check

Using the data_kernel Python API to check GPU hardware and software availability.


In [ ]:
# Check GPU availability
gpu_available = is_gpu_available()
gpu_info = get_gpu_info()

print("=" * 70)
print("GPU AVAILABILITY CHECK")
print("=" * 70)

if gpu_available and gpu_info:
    print(f"Status: ✓ GPU Available")
    print(f"\nGPU Information:")
    print(f"  Device Name:    {gpu_info.get('name', 'Unknown')}")
    print(f"  Backend:        {gpu_info.get('backend', 'Unknown')}")
    print(f"  Device Type:    {gpu_info.get('device_type', 'Unknown')}")
    print(f"  Driver:         {gpu_info.get('driver', 'Unknown')}")
    if gpu_info.get('driver_info'):
        print(f"  Driver Info:    {gpu_info['driver_info']}")
    print(f"\n✓ GPU acceleration is available for this session")
    print(f"Note: The executor will automatically use GPU when beneficial")
else:
    print(f"Status: ⚠️ GPU Not Available")
    print(f"\n⚠️ Tests will run on CPU only")
    print(f"Note: CPU execution via DataFusion is still highly optimized")

print("=" * 70)


## Verify Test Data


In [ ]:
# Verify data files exist
orders_path = DATA_DIR / "orders_1m.parquet"
items_path = DATA_DIR / "order_items_1m.parquet"

print("Data Files:")
print(f"  Orders: {orders_path.exists() and '✓' or '✗'} {orders_path}")
if orders_path.exists():
    size_mb = orders_path.stat().st_size / 1024 / 1024
    print(f"          Size: {size_mb:.2f} MB")
    print(f"          Table name: 'orders_1m'")

print(f"\n  Items:  {items_path.exists() and '✓' or '✗'} {items_path}")
if items_path.exists():
    size_mb = items_path.stat().st_size / 1024 / 1024
    print(f"          Size: {size_mb:.2f} MB")
    print(f"          Table name: 'order_items_1m'")

if not (orders_path.exists() and items_path.exists()):
    raise FileNotFoundError("Test data files not found! Run generate-test-data first.")

print("\n✓ All test data files verified")
print("\nNote: Table names in SQL must match parquet filenames (without .parquet)")


## Test Execution Helper Functions


In [ ]:
def run_sql_test(query, description=""):
    """Execute SQL query using data_kernel and measure performance."""
    
    start = time.time()
    try:
        # Execute query through data_kernel
        result_df = execute(query)
        elapsed = time.time() - start
        
        # Get row count
        row_count = len(result_df) if result_df is not None else 0
        
        return {
            'description': description,
            'query': query[:80] + '...' if len(query) > 80 else query,
            'execution_time_ms': elapsed * 1000,
            'row_count': row_count,
            'status': 'SUCCESS',
            'error': None,
            'result_df': result_df
        }
    
    except Exception as e:
        elapsed = time.time() - start
        return {
            'description': description,
            'query': query[:80] + '...' if len(query) > 80 else query,
            'execution_time_ms': elapsed * 1000,
            'row_count': 0,
            'status': 'ERROR',
            'error': str(e),
            'result_df': None
        }

def print_result(result):
    """Pretty print a test result."""
    status_icon = '✓' if result['status'] == 'SUCCESS' else '✗'
    print(f"{status_icon} {result['description']}")
    print(f"   Time: {result['execution_time_ms']:.2f}ms | Rows: {result['row_count']:,}")
    if result['error']:
        print(f"   Error: {result['error'][:150]}")
    print()

print("✓ Helper functions defined")


---
# SQL Aggregation Tests

Running 10 SQL aggregation tests against 1M orders and 5.5M order_items.

All queries are executed through the **data_kernel Python API**, which calls the Rust executor via FFI.


## Test 1: Simple COUNT
Basic row count - tests scan performance.


In [ ]:
result = run_sql_test(
    query="SELECT COUNT(*) FROM orders_1m",
    description="Simple COUNT(*)"
)
results.append(result)
print_result(result)

if result['status'] == 'SUCCESS' and result['result_df'] is not None:
    print("Result:")
    print(result['result_df'])


## Test 2: Multiple Global Aggregates
Tests parallel aggregate computation.


In [ ]:
result = run_sql_test(
    query="SELECT COUNT(*) as count, SUM(amount) as total, AVG(amount) as avg, MIN(amount) as min, MAX(amount) as max FROM orders_1m",
    description="Multiple global aggregates"
)
results.append(result)
print_result(result)

if result['status'] == 'SUCCESS' and result['result_df'] is not None:
    print("Result:")
    print(result['result_df'])


## Test 3: GROUP BY Status (Low Cardinality)
Tests hash aggregation with 4 groups (pending, shipped, delivered, cancelled).


In [ ]:
result = run_sql_test(
    query="SELECT status, COUNT(*) as count, SUM(amount) as total, AVG(amount) as avg_amount FROM orders_1m GROUP BY status",
    description="GROUP BY status (low cardinality)"
)
results.append(result)
print_result(result)

if result['status'] == 'SUCCESS' and result['result_df'] is not None:
    print("Result by status:")
    print(result['result_df'])


## Test 4: GROUP BY Customer ID (High Cardinality)
Tests hash aggregation with ~1000 groups.


In [ ]:
result = run_sql_test(
    query="SELECT customer_id, COUNT(*) as order_count, SUM(amount) as total_spent, AVG(amount) as avg_order FROM orders_1m GROUP BY customer_id ORDER BY total_spent DESC LIMIT 100",
    description="GROUP BY customer_id (high cardinality)"
)
results.append(result)
print_result(result)

if result['status'] == 'SUCCESS' and result['result_df'] is not None:
    print("Top 10 customers by total spent:")
    print(result['result_df'].head(10))


## Test 5: Filtered Aggregation
Tests filter pushdown with aggregation.


In [ ]:
result = run_sql_test(
    query="SELECT COUNT(*) as high_value_orders, SUM(amount) as total_value, AVG(amount) as avg_value FROM orders_1m WHERE amount > 500",
    description="Filtered aggregation (amount > 500)"
)
results.append(result)
print_result(result)

if result['status'] == 'SUCCESS' and result['result_df'] is not None:
    print("High-value orders (>$500):")
    print(result['result_df'])


## Test 6: Filtered GROUP BY
Combines filter and grouping.


In [ ]:
result = run_sql_test(
    query="SELECT status, COUNT(*) as count, AVG(amount) as avg_amount, SUM(quantity) as total_qty FROM orders_1m WHERE quantity > 5 GROUP BY status",
    description="Filtered GROUP BY (quantity > 5)"
)
results.append(result)
print_result(result)

if result['status'] == 'SUCCESS' and result['result_df'] is not None:
    print("Large quantity orders by status:")
    print(result['result_df'])


## Test 7: JOIN with Aggregation
Tests join performance and aggregation across tables.


In [ ]:
result = run_sql_test(
    query="SELECT COUNT(*) as total_items, SUM(i.price * i.quantity) as total_revenue FROM orders_1m o JOIN order_items_1m i ON o.id = i.order_id",
    description="JOIN with aggregation"
)
results.append(result)
print_result(result)

if result['status'] == 'SUCCESS' and result['result_df'] is not None:
    print("Total revenue from all items:")
    print(result['result_df'])


## Test 8: JOIN with GROUP BY
Tests join with grouped aggregation.


In [ ]:
result = run_sql_test(
    query="SELECT o.status, COUNT(i.id) as item_count, SUM(i.price * i.quantity) as revenue FROM orders_1m o JOIN order_items_1m i ON o.id = i.order_id GROUP BY o.status",
    description="JOIN with GROUP BY"
)
results.append(result)
print_result(result)

if result['status'] == 'SUCCESS' and result['result_df'] is not None:
    print("Revenue by order status:")
    print(result['result_df'])


## Test 9: Complex Multi-Table Aggregation
Tests complex join with multiple aggregates and DISTINCT.


In [ ]:
result = run_sql_test(
    query="SELECT o.customer_id, COUNT(DISTINCT i.product_id) as unique_products, COUNT(i.id) as total_items, SUM(i.price * i.quantity) as total_spent FROM orders_1m o JOIN order_items_1m i ON o.id = i.order_id GROUP BY o.customer_id ORDER BY total_spent DESC LIMIT 50",
    description="Complex multi-table aggregation"
)
results.append(result)
print_result(result)

if result['status'] == 'SUCCESS' and result['result_df'] is not None:
    print("Top 10 customers by spending:")
    print(result['result_df'].head(10))


## Test 10: Subquery with Aggregation
Tests nested aggregation (subquery in WHERE clause).


In [ ]:
result = run_sql_test(
    query="SELECT status, COUNT(*) as count, AVG(amount) as avg_amount FROM orders_1m WHERE amount > (SELECT AVG(amount) FROM orders_1m) GROUP BY status",
    description="Subquery with aggregation"
)
results.append(result)
print_result(result)

if result['status'] == 'SUCCESS' and result['result_df'] is not None:
    print("Above-average orders by status:")
    print(result['result_df'])


---
# Results Summary


In [ ]:
# Create results DataFrame (excluding result_df column for summary)
summary_results = [{k: v for k, v in r.items() if k != 'result_df'} for r in results]
df_results = pd.DataFrame(summary_results)

print("\n" + "=" * 100)
print("SQL AGGREGATION TEST RESULTS")
print("=" * 100)
print(f"\nDataset: 1M orders, 5.5M order_items")
print(f"Execution Engine: DataFusion (Rust) via data_kernel Python API")
print(f"GPU Available: {gpu_available}")
print(f"\nTotal Tests: {len(results)}")
print(f"Successful: {(df_results['status'] == 'SUCCESS').sum()}")
print(f"Failed: {(df_results['status'] != 'SUCCESS').sum()}")

if (df_results['status'] == 'SUCCESS').any():
    successful = df_results[df_results['status'] == 'SUCCESS']
    print(f"\nExecution Time Statistics (ms):")
    print(f"  Mean:   {successful['execution_time_ms'].mean():.2f}")
    print(f"  Median: {successful['execution_time_ms'].median():.2f}")
    print(f"  Min:    {successful['execution_time_ms'].min():.2f}")
    print(f"  Max:    {successful['execution_time_ms'].max():.2f}")
    print(f"  Total:  {successful['execution_time_ms'].sum():.2f}")

print("\n" + "=" * 100)
print("DETAILED RESULTS")
print("=" * 100)

# Display results table
display_df = df_results[['description', 'execution_time_ms', 'row_count', 'status']].copy()
display_df.columns = ['Test', 'Time (ms)', 'Rows', 'Status']
display_df['Time (ms)'] = display_df['Time (ms)'].round(2)
display_df.index = range(1, len(display_df) + 1)

print(display_df.to_string())

# Show any errors
if (df_results['status'] != 'SUCCESS').any():
    print("\n" + "=" * 100)
    print("ERRORS")
    print("=" * 100)
    failed = df_results[df_results['status'] != 'SUCCESS']
    for idx, row in failed.iterrows():
        print(f"\nTest: {row['description']}")
        print(f"Query: {row['query']}")
        print(f"Error: {row['error']}")

print("\n" + "=" * 100)


## Performance Visualization


In [ ]:
try:
    import matplotlib.pyplot as plt
    import numpy as np
    
    # Filter successful tests
    viz_df = df_results[df_results['status'] == 'SUCCESS'].copy()
    
    if len(viz_df) > 0:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
        
        # Execution time bar chart
        ax1.barh(range(len(viz_df)), viz_df['execution_time_ms'], color='steelblue')
        ax1.set_yticks(range(len(viz_df)))
        ax1.set_yticklabels([f"{i+1}. {d[:30]}..." if len(d) > 30 else f"{i+1}. {d}" 
                             for i, d in enumerate(viz_df['description'])], fontsize=9)
        ax1.set_xlabel('Execution Time (ms)', fontsize=11)
        ax1.set_title('SQL Query Performance (data_kernel)', fontsize=13, fontweight='bold')
        ax1.grid(axis='x', alpha=0.3)
        
        # Add value labels
        for i, v in enumerate(viz_df['execution_time_ms']):
            ax1.text(v, i, f' {v:.1f}ms', va='center', fontsize=8)
        
        # Query type categorization
        categories = []
        for desc in viz_df['description']:
            if 'JOIN' in desc:
                categories.append('JOIN')
            elif 'GROUP BY' in desc:
                categories.append('GROUP BY')
            elif 'Filtered' in desc:
                categories.append('Filter')
            else:
                categories.append('Simple')
        
        viz_df['category'] = categories
        category_stats = viz_df.groupby('category')['execution_time_ms'].agg(['mean', 'count'])
        
        colors = {'Simple': '#2ecc71', 'Filter': '#3498db', 'GROUP BY': '#f39c12', 'JOIN': '#e74c3c'}
        bar_colors = [colors.get(cat, 'gray') for cat in category_stats.index]
        
        bars = ax2.bar(range(len(category_stats)), category_stats['mean'], color=bar_colors, alpha=0.7)
        ax2.set_xticks(range(len(category_stats)))
        ax2.set_xticklabels([f"{cat}\n(n={int(row['count'])})" 
                             for cat, row in category_stats.iterrows()], fontsize=10)
        ax2.set_ylabel('Average Execution Time (ms)', fontsize=11)
        ax2.set_title('Performance by Query Type', fontsize=13, fontweight='bold')
        ax2.grid(axis='y', alpha=0.3)
        
        # Add value labels on bars
        for bar in bars:
            height = bar.get_height()
            ax2.text(bar.get_x() + bar.get_width()/2., height,
                    f'{height:.1f}ms',
                    ha='center', va='bottom', fontsize=10, fontweight='bold')
        
        plt.tight_layout()
        plt.show()
        
        print("✓ Performance visualization generated")
    else:
        print("⚠️ No successful tests to visualize")

except ImportError:
    print("⚠️ matplotlib not available - skipping visualization")
    print("   Install with: pip install matplotlib")


## Conclusions

### Key Findings:
- The Rust-based executor (via data_kernel Python API) successfully handles large-scale aggregations on 1M+ rows
- DataFusion provides excellent performance for complex SQL queries
- JOIN operations show good performance even with multi-million row datasets
- GROUP BY operations scale well with different cardinalities
- Python API provides seamless integration with Rust execution engine via FFI

### Performance Characteristics:
1. **Simple aggregations** (COUNT, SUM) are fastest
2. **GROUP BY** adds overhead but remains efficient
3. **JOINs** are the most expensive operations
4. **Filter pushdown** effectively reduces computation
5. **Subqueries** are optimized by DataFusion's query planner

### Technical Architecture:
- **Python Layer**: data_kernel package provides Pythonic API
- **FFI Bridge**: C extension (arrow_bridge) connects Python to Rust
- **Execution Engine**: Rust executor using DataFusion + WGPU
- **Data Format**: Apache Arrow for zero-copy data transfer
- **Storage**: Parquet files for efficient columnar storage

### Next Steps:
- GPU optimization for large aggregations (currently CPU-optimized)
- Testing with larger datasets (10M, 100M rows)
- Comparing with other SQL engines (DuckDB, Polars)
- Benchmarking memory usage and I/O patterns
- Exploring streaming execution for very large datasets
